In [1]:
import joblib
import numpy as np

def load_components():
    judge = joblib.load('../data/surrogate_ids_ctu13.pkl')
    proto_encoder = joblib.load('../data/label_encoder_proto.pkl')
    state_encoder = joblib.load('../data/label_encoder_state.pkl')
    return judge, proto_encoder, state_encoder

def extract_features(sample, proto_enc, state_enc):
    try:
        p_enc = float(proto_enc.transform([str(sample['proto'])])[0])
    except:
        p_enc = 0.0
        
    try:
        s_enc = float(state_enc.transform([str(sample['state'])])[0])
    except:
        s_enc = 0.0
        
    features = np.array([
        float(sample['dur']),
        float(sample['tot_pkts']),
        float(sample['tot_bytes']),
        float(sample['src_bytes']),
        p_enc,
        s_enc
    ], dtype=np.float32)
    return features.reshape(1, -1)

if __name__ == "__main__":
    judge, p_enc, s_enc = load_components()
    
    # Giả lập mẫu Botnet CTU-13 gốc (Thời lượng dài, dung lượng lớn, UDP)
    base_sample = {
        'dur': 15.5,
        'tot_pkts': 100,
        'tot_bytes': 85000,
        'src_bytes': 45000,
        'proto': 'udp',
        'state': 'INT'
    }
    
    base_feats = extract_features(base_sample, p_enc, s_enc)
    base_pred = int(judge.predict(base_feats)[0])
    print("="*50)
    print(f"[*] BASELINE (Không đột biến): Dự đoán = {'Botnet' if base_pred == 1 else 'Normal'}")
    
    # Test 1: a0 (Chỉ cộng thêm Jitter thời gian)
    test1 = base_sample.copy()
    test1['dur'] += 5.0
    pred1 = int(judge.predict(extract_features(test1, p_enc, s_enc))[0])
    print(f"[a0] Chỉ tăng Timing (+5s)           -> {'Botnet' if pred1 == 1 else 'Normal'}")
    
    # Test 2: a1 (Chỉ áp dụng Byte Delta - Padding)
    test2 = base_sample.copy()
    test2['tot_bytes'] += 500.0
    test2['src_bytes'] += 500.0
    test2['tot_pkts'] += 5
    pred2 = int(judge.predict(extract_features(test2, p_enc, s_enc))[0])
    print(f"[a1] Chỉ tăng Byte Delta (+500b)    -> {'Botnet' if pred2 == 1 else 'Normal'}")
    
    # Test 3: a2 (Chỉ nhảy Protocol)
    test3 = base_sample.copy()
    test3['proto'] = 'tcp'
    pred3 = int(judge.predict(extract_features(test3, p_enc, s_enc))[0])
    print(f"[a2] Chỉ nhảy Protocol (-> TCP)     -> {'Botnet' if pred3 == 1 else 'Normal'}")
    
    # Test 4: a3 (Chỉ nhảy State)
    test4 = base_sample.copy()
    test4['state'] = 'CON'
    pred4 = int(judge.predict(extract_features(test4, p_enc, s_enc))[0])
    print(f"[a3] Chỉ nhảy State (-> CON)        -> {'Botnet' if pred4 == 1 else 'Normal'}")
    
    # Test 5: Tổ hợp a1 + a2 + a3 (Phân mảnh + Hop Protocol + Hop State)
    test5 = base_sample.copy()
    test5['tot_bytes'] += 150.0  # Overhead
    test5['src_bytes'] += 150.0
    test5['tot_pkts'] += 20      # Packetization tăng vọt
    test5['proto'] = 'tcp'
    test5['state'] = 'CON'
    pred5 = int(judge.predict(extract_features(test5, p_enc, s_enc))[0])
    print(f"[Tổ hợp] Phân mảnh + TCP + CON      -> {'Botnet (Thất bại)' if pred5 == 1 else 'Normal (Evasion Thành công)'}")
    print("="*50)

[*] BASELINE (Không đột biến): Dự đoán = Botnet
[a0] Chỉ tăng Timing (+5s)           -> Botnet
[a1] Chỉ tăng Byte Delta (+500b)    -> Botnet
[a2] Chỉ nhảy Protocol (-> TCP)     -> Normal
[a3] Chỉ nhảy State (-> CON)        -> Botnet
[Tổ hợp] Phân mảnh + TCP + CON      -> Botnet (Thất bại)


/home/phatkhongfat/anaconda3/envs/rl_c2_evasion/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/phatkhongfat/anaconda3/envs/rl_c2_evasion/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/phatkhongfat/anaconda3/envs/rl_c2_evasion/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/phatkhongfat/anaconda3/envs/rl_c2_evasion/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/phatkhongfat/anaconda3/envs/rl_c2_evasion/lib/python3.